# QLoRA + SFT 原理讲解

本 notebook 用最小的模型（GPT-2）和最精简的代码，带你理解 **QLoRA + SFT（监督微调）** 的完整流程。

## 核心概念

| 概念 | 说明 |
|------|------|
| **SFT** | Supervised Fine-Tuning，用标注数据对预训练模型做监督微调 |
| **LoRA** | Low-Rank Adaptation，只训练两个小矩阵 A、B，冻结原始权重 |
| **QLoRA** | Quantized LoRA，将原始权重量化为 4-bit 节省显存，LoRA 适配器仍用 fp16/bf16 训练（具体取决于 GPU） |

## 架构图

```
原始权重 W (冻结, 4-bit 量化)
      ↓
   量化/反量化
      ↓
  W_dequant (fp16/bf16)
      +
  B @ A (LoRA, 可训练, r << d)
      ↓
   输出 = x(W + BA)
```

**为什么有效？**
- 大模型的参数更新量 ΔW 通常是低秩的（Aghajanyan et al. 2021）
- 用 r=8 的 LoRA 可以捕获 99% 以上的有效梯度方向
- 4-bit 量化让 70B 模型在单张 48GB GPU 上可训练

## Step 0：安装依赖

In [ ]:
# Kaggle / Colab 一键安装
import subprocess, sys

pkgs = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",  # 4-bit 量化核心
    "datasets>=2.18.0",
    "accelerate>=0.29.0",
    "trl>=0.8.0",            # SFT Trainer
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ 依赖安装完毕")

## Step 1：理解 LoRA —— 从零手写一个 LoRALinear

LoRA 的本质：
$$h = xW^T + \underbrace{x B^T A^T}_{\text{LoRA 旁路}}$$

其中 $A \in \mathbb{R}^{r \times d_{in}}$，$B \in \mathbb{R}^{d_{out} \times r}$，$r \ll \min(d_{in}, d_{out})$

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """手写 LoRA，方便理解原理"""
    def __init__(self, original_linear: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.original = original_linear
        self.original.weight.requires_grad_(False)  # 冻结原始权重

        d_in  = original_linear.in_features
        d_out = original_linear.out_features
        self.scaling = alpha / r

        # LoRA 两个小矩阵（参数量 = r*(d_in+d_out)，远小于 d_in*d_out）
        self.lora_A = nn.Parameter(torch.randn(r, d_in) * 0.02)   # 随机初始化
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))          # 零初始化 → 训练开始时 ΔW=0

    def forward(self, x):
        # 原始路径 + LoRA 旁路
        base_out = self.original(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

    def extra_repr(self):
        r = self.lora_A.shape[0]
        d_in, d_out = self.lora_A.shape[1], self.lora_B.shape[0]
        total = d_in * d_out
        lora  = r * (d_in + d_out)
        return f"rank={r}, params: {total:,} → {lora:,} ({lora/total*100:.1f}%)"

# 演示参数压缩比
linear = nn.Linear(768, 768)
lora_linear = LoRALinear(linear, r=8)
print(lora_linear)

trainable = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
total     = sum(p.numel() for p in lora_linear.parameters())
print(f"\n可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## Step 2：理解 4-bit 量化（QLoRA 的 Q）

### 什么是量化？用停车位打个比方

神经网络的权重是高精度浮点数（fp32），存储一个权重需要 32 个 bit。
**量化**就是：把高精度数字压缩成低精度，用更少的 bit 来存储。

4-bit 量化 = 只用 **4 个 bit** = 只有 **2⁴ = 16 个不同的值** 可以表示。

这就像把一条马路的所有停车位从无限多压缩成只剩 16 个——
每辆车（每个权重）只能停到最近的停车位上。

### 关键问题：这 16 个停车位放在哪里？

```
方案A——均匀量化（Uniform）：等间距排开
  |----|----|----|----|----|----|----|----|
 -1                  0                  1

  缺点：神经网络权重大多数堆在 0 附近（两头很少），
        两端的很多停车位几乎没有车停，白白浪费！

方案B——NF4（NormalFloat4）：哪里车多，停车位就密一点
  ||||||||||  |  |   |    |      |         |
 -1          0                             1

  优点：0 附近格点密集 → 大多数权重能被精确表示
        两端格点稀疏  → 少数极端权重精度差点也没关系
```

**为什么权重集中在 0 附近？** 因为权重近似**正态分布**（钟形曲线），下面我们用真实 GPT-2 权重来验证。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy import stats
from scipy.stats import norm as scipy_norm
from transformers import AutoModelForCausalLM

# ══════════════════════════════════════════════════════════════
# 【2a】验证：GPT-2 真实权重是不是集中在 0 附近（正态分布）？
# ══════════════════════════════════════════════════════════════

print("Loading GPT-2 to inspect weight distributions...\n")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")

layers = {
    "Attn Q/K/V":    gpt2.transformer.h[0].attn.c_attn.weight.detach().float(),
    "FFN up-proj":   gpt2.transformer.h[0].mlp.c_fc.weight.detach().float(),
    "FFN down-proj": gpt2.transformer.h[0].mlp.c_proj.weight.detach().float(),
    "Token Embed":   gpt2.transformer.wte.weight[:500].detach().float(),
}

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

print(f"{'Layer':<18}  {'Mean':>8}  {'Std':>8}  {'Skew':>7}  {'Kurt':>7}  Result")
print(f"  Skew->0 = symmetric  |  Kurt->0 = normal-like tails")
print("-" * 72)

for ax, (name, w) in zip(axes, layers.items()):
    flat = w.flatten().numpy()
    mean, std = flat.mean(), flat.std()
    skew = stats.skew(flat)
    kurt = stats.kurtosis(flat)

    ax.hist(flat, bins=70, density=True, alpha=0.55, color="steelblue", label="Actual weights")
    x = np.linspace(mean - 4*std, mean + 4*std, 300)
    ax.plot(x, scipy_norm.pdf(x, mean, std), "r-", lw=2, label="Normal fit")
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel("Weight value", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)

    verdict = "Normal OK" if abs(skew) < 0.5 and abs(kurt) < 2.0 else "Deviates"
    print(f"  {name:<16}  {mean:>8.4f}  {std:>8.4f}  {skew:>7.3f}  {kurt:>7.3f}  {verdict}")

print()
print("结论: GPT-2 各层权重集中在 0 附近，形状接近正态分布（红线）。")
print("-> NF4 量化正是为这种分布设计的，所以效果好。")

plt.suptitle("GPT-2 Weight Distributions  (blue=actual, red=normal fit)", fontsize=11)
plt.tight_layout()
plt.savefig("weight_distribution.png", dpi=100, bbox_inches="tight")
plt.show()
del gpt2

# NF4 官方量化点（bitsandbytes 源码硬编码值），后续 cell 直接用
NF4_OFFICIAL = torch.tensor([
    -1.0,    -0.6962, -0.5251, -0.3949, -0.2844, -0.1848,
    -0.0911,  0.0,     0.0796,  0.1609,  0.2461,  0.3379,
     0.4407,  0.5626,  0.7229,  1.0
])
print(f"\nNF4_OFFICIAL defined: {len(NF4_OFFICIAL)} grid points (used in later cells)")


In [ ]:
# ══════════════════════════════════════════════════════════════
# 【2b】NF4 的 16 个格点是怎么计算出来的？
# ══════════════════════════════════════════════════════════════
#
# 核心思路：把正态分布等概率切成 16 份，取每份的"中点"作为格点。
#
# 类比：100 人按身高排队，分 16 组（每组约 6 人），
#       取每组正中间那人的身高作为这组的代表值。
#
# 数学：第 k 个格点 = 正态分布的 (k+0.5)/16 分位数，再归一化到 [-1,1]

print("NF4 Grid Construction  (equal-probability slicing of N(0,1))")
print("=" * 65)
print()

N = 16
probs  = [(i + 0.5) / N for i in range(N)]
raw    = scipy_norm.ppf(probs)       # 正态分布逆 CDF（分位函数）
theory = raw / abs(raw).max()        # 归一化到 [-1, 1]

print(f"  {'No.':>4}  {'Prob':>8}  {'Grid pt':>9}  ASCII axis (| = this grid point position)")
print("  " + "-" * 62)

for i, (p, t) in enumerate(zip(probs, theory)):
    pos  = int((t + 1) / 2 * 39)
    axis = ["-"] * 40
    axis[pos] = "|"
    print(f"  [{i:2d}]  p={p:.4f}   {t:+.4f}   {''.join(axis)}")

print()
print("Key observation:")
print("  Points 6 & 7 (at -0.09 and 0.00) have the smallest gap")
print("  -> highest precision near 0, where most weights live")
print("  Points 0 & 15 (at -1.0 and +1.0) are at the extremes")
print("  -> lower precision there, but very few weights are that large")

# 可视化：两种方案对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.linspace(-3.2, 3.2, 800)
pdf = scipy_norm.pdf(x)

for ax, (pts, title, color) in zip(axes, [
    (np.linspace(-1, 1, 16), "Uniform Quantization\n(equal spacing -- wastes grids at tails)", "orange"),
    (theory,                  "NF4 Quantization\n(equal probability -- dense near zero)",       "steelblue"),
]):
    ax.fill_between(x, pdf, alpha=0.12, color="gray", label="Weight distribution")
    ax.plot(x, pdf, color="gray", lw=1.5)
    for p in pts:
        ax.axvline(p, color=color, alpha=0.75, lw=1.5)
    ax.scatter(pts, [0.02] * N, color=color, s=60, zorder=5, label="Grid points")

    n_center = sum(1 for p in pts if abs(p) < 0.5)
    ax.text(0.03, 0.88,
            f"Within |x|<0.5: {n_center}/16 pts\nOuter region:  {16-n_center}/16 pts",
            transform=ax.transAxes, fontsize=10, color=color,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.88))
    ax.set_xlim(-2.8, 2.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Normalized weight value")
    ax.set_ylabel("Probability density  (higher = more weights here)")
    ax.legend(fontsize=9)

plt.suptitle("Grid Placement: Uniform vs NF4\n"
             "Gray curve = weight distribution; denser grids = more precision there", fontsize=12)
plt.tight_layout()
plt.savefig("nf4_points.png", dpi=100, bbox_inches="tight")
plt.show()


### 量化完整四步流程：fp32 → 4-bit → fp32

```
原始权重（fp32，每个值 4 bytes，精度很高）
       │
       ▼ ① 把 64 个权重分为一块，找出这块里的最大绝对值（叫 scale）
     scale = max(|w₁|, |w₂|, ..., |w₆₄|)    ← scale 单独保存用于还原
       │
       ▼ ② 所有权重除以 scale，压缩到 [-1, 1] 范围
     w_norm = w / scale
       │
       ▼ ③ 从 NF4 的 16 个格点里找最近的一个，记录编号（0~15）
     index = argmin|w_norm - 格点|             ← 只存 4-bit 编号
       │
       ▼ ④ 反量化（推理时用格点值还原近似权重）
     w_approx = NF4格点[index] × scale
```

**为什么要分块（block_size=64）而不是整层用一个 scale？**
如果整层只有一个 scale，一个异常大的权重会让 scale 变得很大，
导致其他普通权重压缩到 [-1,1] 后都堆在 0 附近，16 个格点里大部分用不上。
分块后每块独立计算 scale，局部异常值只影响本块，整体精度更好。

**双重量化（Double Quantization）**：
scale 本身也要存储（fp32 = 4 bytes/块）。
QLoRA 进一步把 scale 也量化成 8-bit（1 byte/块），额外省了约 **0.37 bit/param**。
这就是"双重量化"的含义：先量化权重，再量化量化时用的 scale。

In [ ]:
from matplotlib.lines import Line2D

# ══════════════════════════════════════════════════════════════
# 【2c】完整量化流程演示 + NF4 vs 均匀量化 误差对比
# 注意：请先运行上面的 cell 2a，NF4_OFFICIAL 在那里定义。
# ══════════════════════════════════════════════════════════════

BLOCK_SIZE = 64

def quantize_and_dequant(weights, points):
    # Block quantize: split into blocks of 64, each with its own scale
    # Steps: absmax scale -> normalize to [-1,1] -> lookup nearest grid -> dequantize
    n = len(weights)
    dequantized = torch.zeros_like(weights)
    scales = []
    for i in range(0, n, BLOCK_SIZE):
        block  = weights[i : i + BLOCK_SIZE]
        scale  = block.abs().max().clamp(min=1e-8)                    # 1. absmax
        w_norm = block / scale                                          # 2. normalize
        pts    = torch.tensor(points, dtype=block.dtype)
        idx    = (w_norm.unsqueeze(-1) - pts).abs().argmin(dim=-1)    # 3. lookup
        dequantized[i : i + BLOCK_SIZE] = pts[idx] * scale            # 4. dequantize
        scales.append(scale.item())
    return dequantized, scales

torch.manual_seed(0)
N = 4096
weights = torch.randn(N) * 0.02    # 模拟正态分布权重（均值=0，std=0.02）

print(f"Simulated weights: {N} params,  mean={weights.mean():.5f},  std={weights.std():.5f}")
print(f"Block size: {BLOCK_SIZE},  num blocks: {N//BLOCK_SIZE}\n")

dq_uni, _ = quantize_and_dequant(weights, list(np.linspace(-1, 1, 16)))
dq_nf4, _ = quantize_and_dequant(weights, NF4_OFFICIAL.tolist())

err_uni = (weights - dq_uni).abs()
err_nf4 = (weights - dq_nf4).abs()

def snr_db(orig, err):
    return (10 * torch.log10((orig**2).mean() / (err**2).mean())).item()

# 误差对比表
print("=" * 60)
print("Quantization Error Comparison")
print("=" * 60)
print(f"  {'Metric':<22}  {'Uniform':>12}  {'NF4':>12}  {'NF4 gain':>10}")
print("  " + "-" * 56)
for name, v_u, v_n in [
    ("Mean abs error",    err_uni.mean().item(),      err_nf4.mean().item()),
    ("Max abs error",     err_uni.max().item(),       err_nf4.max().item()),
    ("MSE",               (err_uni**2).mean().item(), (err_nf4**2).mean().item()),
]:
    pct = (v_u - v_n) / v_u * 100
    print(f"  {name:<22}  {v_u:>12.6f}  {v_n:>12.6f}  {pct:>+8.1f}%")
snr_u = snr_db(weights, err_uni)
snr_n = snr_db(weights, err_nf4)
print(f"  {'SNR (dB)':<22}  {snr_u:>12.2f}  {snr_n:>12.2f}  {snr_n-snr_u:>+8.2f} dB")
print(f"\n  (+3 dB SNR ~ error cut in half)")
print(f"  NF4 gains {snr_n-snr_u:.1f} dB for normally-distributed weights")

# 存储开销
n_blocks = N // BLOCK_SIZE
fp32_b, nf4_b, nf4dq_b = N*4, N*0.5+n_blocks*4, N*0.5+n_blocks*1
print(f"\n  Storage (this {N}-param example):")
print(f"    fp32 baseline:       {fp32_b:5d} B  (100%)")
print(f"    NF4 + fp32 scale:    {nf4_b:5.0f} B  ({nf4_b/fp32_b*100:.1f}%)")
print(f"    NF4 + double quant:  {nf4dq_b:5.0f} B  ({nf4dq_b/fp32_b*100:.1f}%)")
print(f"  Scaled to 7B model: fp32 ~28 GB  ->  NF4+DQ ~3.9 GB")

# 四张可视化图
uniform_pts = list(np.linspace(-1, 1, 16))
nf4_pts     = NF4_OFFICIAL.tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 图1：weight value vs error scatter
ax = axes[0, 0]
idx = torch.randperm(N)[:600]
ax.scatter(weights[idx].numpy(), err_uni[idx].numpy(), s=6, alpha=0.4, color="orange",    label="Uniform")
ax.scatter(weights[idx].numpy(), err_nf4[idx].numpy(), s=6, alpha=0.4, color="steelblue", label="NF4")
ax.axvline(0, color="gray", ls="--", alpha=0.4)
ax.set_xlabel("Weight value"); ax.set_ylabel("Quantization error")
ax.set_title("Weight value vs error\n(NF4 smaller near zero where weights cluster)")
ax.legend(markerscale=3)

# 图2：error histograms
ax = axes[0, 1]
ax.hist(err_uni.numpy(), bins=55, alpha=0.6, density=True, color="orange",
        label=f"Uniform  mean={err_uni.mean():.5f}")
ax.hist(err_nf4.numpy(), bins=55, alpha=0.6, density=True, color="steelblue",
        label=f"NF4      mean={err_nf4.mean():.5f}")
ax.set_xlabel("Error magnitude"); ax.set_ylabel("Density")
ax.set_title("Error distribution\n(NF4 errors concentrated at smaller values)")
ax.legend()

# 图3：per-bin average error
ax = axes[1, 0]
bins = np.linspace(weights.min().item(), weights.max().item(), 21)
centers = (bins[:-1] + bins[1:]) / 2
uni_m = [err_uni[(weights>=bins[i])&(weights<bins[i+1])].mean().item() for i in range(20)]
nf4_m = [err_nf4[(weights>=bins[i])&(weights<bins[i+1])].mean().item() for i in range(20)]
ax.plot(centers, uni_m, "o-", color="orange",    label="Uniform", ms=5)
ax.plot(centers, nf4_m, "s-", color="steelblue", label="NF4",     ms=5)
ax.axvline(0, color="gray", ls="--", alpha=0.4)
ax.set_xlabel("Weight value (bin center)"); ax.set_ylabel("Mean error in bin")
ax.set_title("Per-bin mean error\n(NF4 advantage largest near zero)")
ax.legend()

# 图4：grid point positions on normal curve
ax = axes[1, 1]
x = np.linspace(-3, 3, 500)
ax.fill_between(x, scipy_norm.pdf(x), alpha=0.12, color="gray")
ax.plot(x, scipy_norm.pdf(x), "gray", lw=1.5)
for p in nf4_pts:     ax.axvline(p, color="steelblue", alpha=0.7, lw=1.3)
for p in uniform_pts: ax.axvline(p, color="orange",    alpha=0.5, lw=1.0, ls="--")
ax.scatter(nf4_pts,     [0.014]*16, color="steelblue", s=45, zorder=5)
ax.scatter(uniform_pts, [0.005]*16, color="orange",    s=45, zorder=5, marker="^")
ax.legend(handles=[
    Line2D([0],[0], color="steelblue", lw=2, label="NF4 (dense near zero)"),
    Line2D([0],[0], color="orange",    lw=2, ls="--", label="Uniform (evenly spaced)"),
    Line2D([0],[0], color="gray",      lw=2, label="Weight distribution"),
], fontsize=9)
ax.set_xlim(-2.6, 2.6)
ax.set_title("Grid point positions\nHigher curve = more weights; denser grids = more precision")

plt.suptitle("4-bit Quantization Full Analysis: Uniform vs NF4", fontsize=13)
plt.tight_layout()
plt.savefig("nf4_full_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
print("Quantization analysis complete")


## Step 3：加载量化模型 + 挂载 LoRA 适配器

使用 `GPT-2`（最小 124M 参数），演示完整 QLoRA 配置流程。

> 真实生产中替换为 `meta-llama/Llama-3-8B` 等大模型，其余代码完全相同。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch

MODEL_ID = "gpt2"  # 124M 参数，最小的演示模型

# ── 1. 检测 GPU ────────────────────────────────────────────────────────────
has_cuda = torch.cuda.is_available()
print(f"CUDA 可用: {has_cuda}")
if has_cuda:
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")

# ── 2. 加载 tokenizer ──────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 没有 pad token，用 eos 代替

# ── 3. 加载模型（GPU 走 4-bit 量化，CPU 走 fp32）─────────────────────────
if has_cuda:
    # 4-bit 量化配置
    # 注意：用 float16 而非 bfloat16，Kaggle T4/V100 不支持原生 bf16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,   # T4/V100 用 fp16
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    # prepare_model_for_kbit_training 做三件事：
    # 1) LayerNorm → fp32（保持训练稳定）
    # 2) 开启 gradient checkpointing（节省显存）
    # 3) 冻结所有非 LoRA 参数
    model = prepare_model_for_kbit_training(model)
    quant_label = "4-bit NF4 (fp16 compute)"
else:
    # 没有 GPU 时用 fp32，可以演示 LoRA 逻辑，但无法做 4-bit 量化
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    quant_label = "fp32 (CPU, no quantization)"

print(f"模型加载完毕，模式: {quant_label}")

# ── 4. 配置 LoRA 适配器 ────────────────────────────────────────────────────
# GPT-2 的注意力层叫 c_attn（Q/K/V 合并在一起），这是我们要训练的目标层
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,            # LoRA 秩：越大表达能力越强，显存越多
    lora_alpha=16,  # 缩放系数，实际缩放 = alpha/r = 2
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 预期输出类似：trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.24


## Step 4：准备 SFT 数据集

SFT 数据格式：`{instruction, input, output}` → 拼成单条文本让模型学习预测 output 部分。

这里用一个玩具数据集演示，真实场景可替换为 Alpaca / OpenHermes 等。

In [ ]:
from datasets import Dataset

# ── 玩具 SFT 数据（instruction-following 格式）──────────────────────────
raw_data = [
    {"instruction": "将下面的句子翻译成英文。", "input": "今天天气很好。",       "output": "The weather is nice today."},
    {"instruction": "计算以下数学题。",          "input": "3 + 5 = ?",           "output": "8"},
    {"instruction": "用一句话描述机器学习。",     "input": "",                   "output": "Machine learning is a field of AI that enables computers to learn from data."},
    {"instruction": "列出3种水果。",             "input": "",                   "output": "Apple, banana, and orange."},
    {"instruction": "将下面的句子翻译成英文。",   "input": "我喜欢编程。",         "output": "I love programming."},
    {"instruction": "解释什么是神经网络。",       "input": "",                   "output": "A neural network is a computational model inspired by the human brain."},
    {"instruction": "计算以下数学题。",          "input": "12 * 4 = ?",          "output": "48"},
    {"instruction": "将下面的句子翻译成英文。",   "input": "深度学习很强大。",     "output": "Deep learning is very powerful."},
]

# ── Alpaca 格式 prompt 模板 ───────────────────────────────────────────────
PROMPT_TEMPLATE = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_sample(sample):
    return {"text": PROMPT_TEMPLATE.format(**sample)}

dataset = Dataset.from_list([format_sample(s) for s in raw_data])

# 查看一条样本
print(dataset[0]["text"])
print(f"\n数据集大小: {len(dataset)} 条")

## Step 5：SFT 训练

我们使用 `transformers.Trainer`（HuggingFace 官方标准训练器）来完成 SFT，它兼容所有版本、无需担心 TRL API 变化。

训练前需要先把数据 **预 tokenize**（把文字转成模型能看懂的数字序列）：

```
文本 "The weather is nice"
    ↓ tokenizer
[464, 6193, 318, 3621]   ← input_ids
[464, 6193, 318, 3621]   ← labels（语言模型：预测下一个 token）
```

`DataCollatorForLanguageModeling(mlm=False)` 负责将多条样本拼成一批（batch），
其中 `mlm=False` 表示做 **因果语言模型**（Causal LM，即自回归预测），而非 BERT 的 masked LM。

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# ── Step 1：预 tokenize 数据集 ───────────────────────────────────────────
def tokenize_fn(examples):
    # 只做 tokenize，不设置 labels——DataCollatorForLanguageModeling 会自动从
    # 已 pad 的 input_ids 创建 labels，并把 padding 位置设为 -100（不计入 loss）
    return tokenizer(examples["text"], truncation=True, max_length=256, padding=False)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
print(f"Tokenized dataset: {tokenized_dataset}")
print(f"Sample input_ids length: {len(tokenized_dataset[0]['input_ids'])}")

# ── Step 2：Data Collator（负责 padding 和 batch 拼装）─────────────────
# mlm=False → 因果语言模型（GPT 风格自回归），不是 masked LM（BERT 风格）
# 它自动把每个 batch 里的序列 pad 到相同长度，并把 pad 位置的 label 设为 -100
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ── Step 3：训练参数 ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./qlora_sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # 等效 batch_size = 8
    learning_rate=2e-4,
    fp16=has_cuda,                   # GPU 用 fp16 混合精度，CPU 用 fp32
    logging_steps=5,
    save_strategy="no",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
)

# ── Step 4：启动训练 ─────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("开始训练 ...")
trainer.train()
print("Training complete!")


## Step 6：保存 & 加载 LoRA 适配器

LoRA 的一大优势：只需保存小小的适配器（几 MB），而不是完整模型（几 GB）。

In [ ]:
import os

# 保存 LoRA 适配器（只保存 ΔW，不保存原始权重）
adapter_path = "./qlora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# 查看保存的文件大小
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f))
    print(f"  {f:40s}  {size/1024:.1f} KB")

print("\n适配器很小！真实场景中 LoRA adapter 通常只有 10-100 MB，")
print("而完整的 7B 模型则需要 14 GB（fp16）或 4 GB（4-bit）")

## Step 7：推理测试

In [ ]:
# 方式一：直接用训练后的 model 推理
model.eval()

def generate(instruction, input_text="", max_new_tokens=50):
    prompt = PROMPT_TEMPLATE.format(
        instruction=instruction,
        input=input_text,
        output=""   # 留空，让模型生成
    )
    # 把输入张量放到模型所在设备（CUDA / MPS / CPU 都自动适配）
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 只返回新生成的部分（去掉 prompt）
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# 测试两条 prompt
result1 = generate("计算以下数学题。", "6 + 7 = ?")
print(f"生成结果 1: {result1}")

result2 = generate("用一句话描述深度学习。")
print(f"生成结果 2: {result2}")

print("\n注意: 玩具数据集只有 8 条样本，3 epoch 远远不够让模型学到东西。")
print("真实场景下需要数千~数万条 SFT 数据 + 充分训练。")


## Step 8：合并 LoRA 权重（可选，用于部署）

训练完成后，可以将 LoRA 适配器合并回原始模型，得到一个普通的完整模型用于高效推理。

$$W_{merged} = W_{original} + \frac{\alpha}{r} \cdot BA$$

In [ ]:
# 注意：merge_and_unload 需要在 fp16/fp32 模型上操作（4-bit 量化模型不支持直接 merge）
# 生产流程：训练完后重新加载 fp16 基座 + LoRA adapter → merge → 保存

if not has_cuda:
    # CPU 环境可以直接 merge
    merged_model = model.merge_and_unload()
    print(f"合并后模型类型: {type(merged_model).__name__}")
    print("合并后可以像普通模型一样推理和部署，无需 PEFT 库")
else:
    print("GPU 环境下，合并流程：")
    print("1. 重新以 fp16 加载基座模型")
    print("2. base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)")
    print("3. merged = PeftModel.from_pretrained(base, adapter_path).merge_and_unload()")
    print("4. merged.save_pretrained('./merged_model')")

## 总结：QLoRA + SFT 完整流程

```
预训练模型
    │
    ▼ BitsAndBytesConfig (load_in_4bit=True)
量化基座 (4-bit NF4)
    │
    ▼ prepare_model_for_kbit_training()
冻结原始权重 + 开启梯度检查点
    │
    ▼ get_peft_model(model, LoraConfig(r=8))
注入 LoRA 适配器 (只有 ~0.1% 参数可训练)
    │
    ▼ Trainer.train()
在 instruction 数据上监督微调
    │
    ▼ model.save_pretrained()
保存小小的 LoRA adapter (~10MB vs ~4GB)
    │
    ▼ [可选] merge_and_unload()
合并权重，得到完整微调模型用于部署
```

### 关键超参数选择指南

| 超参数 | 推荐值 | 说明 |
|--------|--------|------|
| `r` | 8~64 | 越大效果越好，显存/计算越多 |
| `lora_alpha` | 2×r | 缩放比 α/r 通常取 1~2 |
| `target_modules` | attention layers | Q/K/V/O 都加效果最好 |
| `learning_rate` | 1e-4 ~ 3e-4 | 比全参数微调大 10x |
| `lora_dropout` | 0.05 | 小数据集可适当增大 |
